# Five-method upstream reference checks

**HiDDeN, RoSteALS, SepWater:** freshly reload the selected upstream neural core and compare it with the frozen adapter. Same five historical crops, five full-length messages each, weights and locked dependencies. Both decoders also receive the other branch's exact pixels.

**SepMark, DistSeal:** test source-justified export rules on the historical five images and the next five from the frozen manifest. This part is a serializer check, not an independently reimplemented encoder.

No attacks, matched PSNR, training, payload shortening or production edits. Results go into a new Drive folder. A finished test may still have nonzero BER.

Use **A100 + high RAM** as in the previous diagnostic; model environments and phases run sequentially. Real GPU execution of this release still needs this Colab test. Upload the matching **Embedding_Bank_AUTHOR_REFERENCE_source.zip** when asked; it contains readable source, no Base64 or model weights.

## 1. Settings

Keep N_IMAGES=5. BANK_ROOT is the existing native-resolution canary with actual PNGs and metadata, not a report ZIP. Do not point OUTPUT_ROOT at previous results. The next five original images must still be available in the bank.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, io

N_IMAGES = 5                 # Fixed scope; do not change to 100 in this notebook.
MODELS = ["hidden_benchmark", "rosteals", "sepwater", "sepmark", "distseal_posthoc_dcae"]              # Only these five diagnostic methods.
DEVICE = "cuda"
MAX_HOURS = 6              # Safety ceiling, not a duration estimate or Colab guarantee.
SOURCE_ARCHIVE = Path("Embedding_Bank_AUTHOR_REFERENCE_source.zip")
DOWNLOAD_AUTHOR_ASSETS = True
MANUAL_ASSETS_ROOT = ""     # Optional directory with <method>/<checkpoint>.

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BANK_ROOT = Path("/content/drive/MyDrive/watermarking_native_resolution_canary_v1")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/watermarking_author_reference_v1")
    SCRATCH_ROOT = Path("/content/watermarking_author_reference_scratch_v1")
else:
    BANK_ROOT = Path("E:/watermarking_native_resolution_canary_v1")
    OUTPUT_ROOT = Path("E:/watermarking_author_reference_v1")
    SCRATCH_ROOT = Path.home() / "watermarking_author_reference_scratch_v1"

SETTINGS = dict(bank=str(BANK_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT),
                n_images=N_IMAGES, models=MODELS, device=DEVICE, max_hours=MAX_HOURS,
                cleanup=True, download_author_assets=DOWNLOAD_AUTHOR_ASSETS,
                manual_assets_root=MANUAL_ASSETS_ROOT or str(BANK_ROOT / "_assets"))
print("Read-only originals:", BANK_ROOT)
print("NEW durable results:", OUTPUT_ROOT)
print("125 historical cases + 50 new serializer-validation cases. No attacks or PSNR matching.")


## 2. Drive and readable source package

The source ZIP is cached on Drive after its first upload. Each method uses its old dependency lock. Only the lightweight controller is installed in the notebook kernel.

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive is not mounted")
if not (BANK_ROOT / "_inputs/manifest.json").is_file():
    raise FileNotFoundError(f"Set BANK_ROOT to the old bank containing _inputs/manifest.json: {BANK_ROOT}")
for left, right in ((BANK_ROOT, OUTPUT_ROOT), (BANK_ROOT, SCRATCH_ROOT), (OUTPUT_ROOT, SCRATCH_ROOT)):
    if left.resolve().is_relative_to(right.resolve()) or right.resolve().is_relative_to(left.resolve()):
        raise ValueError("Old bank, output and scratch must be separate, non-nested directories")
if not shutil.which("git"):
    raise RuntimeError("Git is required for the pinned author sources")
if DEVICE == "cuda" and not shutil.which("nvidia-smi"):
    raise RuntimeError("Select an NVIDIA GPU runtime first")
if (OUTPUT_ROOT / "bank.json").exists():
    raise ValueError("OUTPUT_ROOT points to a legacy bank. Use a NEW folder.")

SOURCE_SHA256 = "1d5c1a7b641e2b0a01d3841d86cb2050a369c680940013cb6a14aebd37ea7f3a"
CACHED_ARCHIVE = OUTPUT_ROOT / "_release" / (SOURCE_SHA256 + ".zip")
archive_path = SOURCE_ARCHIVE if SOURCE_ARCHIVE.is_file() else CACHED_ARCHIVE
payload = archive_path.read_bytes() if archive_path.is_file() else None
if IN_COLAB and (payload is None or hashlib.sha256(payload).hexdigest() != SOURCE_SHA256):
    from google.colab import files
    print("Upload Embedding_Bank_AUTHOR_REFERENCE_source.zip supplied with this notebook.")
    uploads = files.upload()
    candidates = [data for data in uploads.values() if hashlib.sha256(data).hexdigest() == SOURCE_SHA256]
    if not candidates:
        raise ValueError("Wrong source ZIP: checksum does not match this notebook")
    payload = candidates[0]
if payload is None or hashlib.sha256(payload).hexdigest() != SOURCE_SHA256:
    raise ValueError("Place the matching ZIP beside the notebook or set SOURCE_ARCHIVE")

if N_IMAGES != 5:
    raise ValueError("This diagnostic uses a fixed five-image replay and next-five holdout")
CACHED_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
if not CACHED_ARCHIVE.exists():
    temporary = CACHED_ARCHIVE.with_suffix(".part")
    temporary.write_bytes(payload)
    temporary.replace(CACHED_ARCHIVE)
PROJECT = SCRATCH_ROOT / "release" / SOURCE_SHA256[:16]
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        target = (PROJECT / item.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or item.is_dir():
            raise ValueError("Unexpected archive member")
        content = archive.read(item)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Release source modified: {target}; use fresh scratch")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

# Only small controller tools enter this kernel. Each model gets a separate venv.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10", str(controller_dir)], check=True)
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER), "numpy==1.26.4",
                "Pillow==10.4.0", "psutil==6.1.1", "gdown==5.2.0"], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(PROJECT / "src"), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4",
           OPENBLAS_NUM_THREADS="1", PATH=str(Path(UV).parent) + os.pathsep + ENV.get("PATH", ""))
CONFIG = SCRATCH_ROOT / "author_reference_settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Free local SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Ready. Stop any previous runner for this output directory before starting.")


## 3. Run checks

Completed case/phase records survive reconnects. Re-run cells 1–3 with the same settings. Source/checkpoint changes or damaged evidence stop the comparison instead of overwriting it. Keep the same GPU/runtime during a paired run; a software change is reported explicitly.

In [ ]:
# After reconnecting: run cells 1-3 again with the SAME paths and source ZIP.
# Finished measurements (including nonzero BER) are reused, not improved by retrying.
command = [str(CONTROLLER), "-m", "wmbench_author_reference.runner", "run", "--settings", str(CONFIG)]
process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding="utf-8", errors="replace", start_new_session=(os.name != "nt"))
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f"Controller exit={code}; completed results remain in {OUTPUT_ROOT}")
except BaseException:
    if process.poll() is None:
        if os.name != "nt":
            import signal
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
    raise
print("Queue finished or time budget reached. Inspect the report, not only this message.")


## 4. Report and review ZIP

You can run this after a partial run. It reads small JSON files and does not load models. Send author_reference_report.zip for review; it excludes weights, source-image and float-tensor files.

In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_author_reference.runner", "report",
                "--settings", str(CONFIG)], env=ENV, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT_ROOT / "_reports/REPORT.md").read_text(encoding="utf-8")))

# Review archive includes diagnostic records/logs, not weights or the large image bank.
review = OUTPUT_ROOT / "author_reference_report.zip"
with zipfile.ZipFile(review, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_ROOT.rglob("*"):
        if not path.is_file() or "_assets" in path.relative_to(OUTPUT_ROOT).parts:
            continue
        if path.suffix.lower() in {".json", ".csv", ".md", ".log", ".txt", ".lock"}:
            archive.write(path, path.relative_to(OUTPUT_ROOT).as_posix())
print("Send this archive for review:", review)
print("Review core agreement and holdout serialization separately. Production and old results are unchanged.")
